# Origin

Scaffolding generator for the ProgFun labs.

1. Set `LABNUM` and run **Cell 1** to create the lab directory tree.
2. Set `EXERCISES` and run **Cell 2** to create the per-exercise files.
3. Run **Cell 3** to drop `all_tests.py` into `lab{LABNUM}/console/`.

## Cell 1 — Directory set-up

In [ ]:
import os

LABNUM = 3  # <-- hyperparameter

ROOT = os.path.abspath("")            # directory holding Origin.ipynb / skeleton.c
LAB = os.path.join(ROOT, f"lab{LABNUM}")
SUBDIRS = ["src", "test", "console", "pdf"]

os.makedirs(LAB, exist_ok=True)
for d in SUBDIRS:
    os.makedirs(os.path.join(LAB, d), exist_ok=True)

print(f"lab{LABNUM}/")
for d in SUBDIRS:
    print(f"    {d}/")

## Cell 2 — Files set-up

In [ ]:
EXERCISES = ["","","","",""]  # <-- hyperparameter

SKELETON = os.path.join(ROOT, "skeleton.c")
N_TESTS = 3

with open(SKELETON) as f:
    skeleton = f.read()


def write(path, content="", executable=False):
    with open(path, "w") as f:
        f.write(content)
    if executable:
        os.chmod(path, 0o755)
    print("    " + os.path.relpath(path, ROOT))


def console_script(exercise):
    """Compile from ../src, diff each ../test input against a line of the answer key."""
    lines = [
        'cd "$(dirname "$0")"',
        f"gcc -std=c99 -Wall -pedantic ../src/{exercise}.c -o {exercise} -lm",
    ]
    for i in range(1, N_TESTS + 1):
        lines.append(
            f'[ "$(./{exercise} < ../test/{exercise}{i}.txt)" '
            f"= \"$(sed -n '{i}p' ../test/{exercise}_answerkey.txt)\" ] "
            f"&& echo True || echo False"
        )
    lines.append(f"rm {exercise}")
    return "\n".join(lines) + "\n"


for exercise in EXERCISES:
    print(f"{exercise}:")
    write(os.path.join(LAB, "src", f"{exercise}.c"), skeleton.replace("FIXME:", exercise))
    for i in range(1, N_TESTS + 1):
        write(os.path.join(LAB, "test", f"{exercise}{i}.txt"))
    write(os.path.join(LAB, "test", f"{exercise}_answerkey.txt"))
    write(os.path.join(LAB, "console", f"{exercise}.sh"), console_script(exercise), executable=True)

mulsum:
    lab2/src/mulsum.c
    lab2/test/mulsum1.txt
    lab2/test/mulsum2.txt
    lab2/test/mulsum3.txt
    lab2/test/mulsum_answerkey.txt
    lab2/console/mulsum.sh
takuzu:
    lab2/src/takuzu.c
    lab2/test/takuzu1.txt
    lab2/test/takuzu2.txt
    lab2/test/takuzu3.txt
    lab2/test/takuzu_answerkey.txt
    lab2/console/takuzu.sh
collision:
    lab2/src/collision.c
    lab2/test/collision1.txt
    lab2/test/collision2.txt
    lab2/test/collision3.txt
    lab2/test/collision_answerkey.txt
    lab2/console/collision.sh
primegaps:
    lab2/src/primegaps.c
    lab2/test/primegaps1.txt
    lab2/test/primegaps2.txt
    lab2/test/primegaps3.txt
    lab2/test/primegaps_answerkey.txt
    lab2/console/primegaps.sh
cyclicmult:
    lab2/src/cyclicmult.c
    lab2/test/cyclicmult1.txt
    lab2/test/cyclicmult2.txt
    lab2/test/cyclicmult3.txt
    lab2/test/cyclicmult_answerkey.txt
    lab2/console/cyclicmult.sh


## Cell 3 — Test runner

Writes `lab{LABNUM}/console/all_tests.py`. Run it with `python3 all_tests.py`
from inside `console/`: it runs every `.sh` next to it and prints a single
`True` / `False` verdict at the bottom (exit code 0 / 1).

In [ ]:
ALL_TESTS = r'''#!/usr/bin/env python3
"""Run every .sh test in this folder and report whether they all passed."""
import glob
import os
import subprocess
import sys

HERE = os.path.dirname(os.path.abspath(__file__))


def main():
    scripts = sorted(glob.glob(os.path.join(HERE, "*.sh")))
    if not scripts:
        print("no .sh tests found in", HERE)
        return 1

    all_ok = True
    for script in scripts:
        name = os.path.basename(script)
        proc = subprocess.run(
            ["bash", script], cwd=HERE, capture_output=True, text=True
        )
        results = [
            line.strip()
            for line in proc.stdout.splitlines()
            if line.strip() in ("True", "False")
        ]
        ok = proc.returncode == 0 and bool(results) and all(r == "True" for r in results)
        all_ok = all_ok and ok

        detail = " ".join(results) if results else "no test output"
        print(f"{name:<24} {ok}   [{detail}]")
        for line in proc.stderr.strip().splitlines():
            print(f"    | {line}")

    print("-" * 48)
    print(all_ok)
    return 0 if all_ok else 1


if __name__ == "__main__":
    sys.exit(main())
'''

write(os.path.join(LAB, "console", "all_tests.py"), ALL_TESTS, executable=True)